# Ноутбук 02: Wrapper (RFE, SFS) и Embedded (L1, важности деревьев, Permutation)

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import json
from itertools import combinations
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from lab_utils import (
    load_dataset, split_xy, train_test_split_stratified, build_preprocessor,
    transform_with_names, append_ranking_rows
)

In [2]:
# Загрузка и препроцессинг
df_med = load_dataset('../data/medical.csv')
X_med, y_med = split_xy(df_med)
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split_stratified(X_med, y_med)

df_fin = load_dataset('../data/finance.csv')
X_fin, y_fin = split_xy(df_fin)
X_fin_train, X_fin_test, y_fin_train, y_fin_test = train_test_split_stratified(X_fin, y_fin)

preprocessor_med = build_preprocessor(X_med_train)
X_med_train_t, X_med_test_t, feat_names_med = transform_with_names(preprocessor_med, X_med_train, X_med_test)

preprocessor_fin = build_preprocessor(X_fin_train)
X_fin_train_t, X_fin_test_t, feat_names_fin = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

rows = []

In [3]:
# RFE
estimator = LogisticRegression(max_iter=1000, random_state=42)
selector_rfe = RFE(estimator, n_features_to_select=10, step=1)
selector_rfe.fit(X_med_train_t, y_med_train)
rfe_scores = 1.0 / (selector_rfe.ranking_ + 1e-6)
append_ranking_rows(rows, 'medical', 'RFE', feat_names_med, rfe_scores)

selector_rfe.fit(X_fin_train_t, y_fin_train)
rfe_scores_fin = 1.0 / (selector_rfe.ranking_ + 1e-6)
append_ranking_rows(rows, 'finance', 'RFE', feat_names_fin, rfe_scores_fin)

In [4]:
# SFS forward
sfs = SequentialFeatureSelector(estimator, n_features_to_select=10, direction='forward', cv=3)
sfs.fit(X_med_train_t, y_med_train)
selected = sfs.get_support()
sfs_scores = np.where(selected, 1.0, 0.0)
append_ranking_rows(rows, 'medical', 'SFS_forward', feat_names_med, sfs_scores)

sfs.fit(X_fin_train_t, y_fin_train)
selected_fin = sfs.get_support()
sfs_scores_fin = np.where(selected_fin, 1.0, 0.0)
append_ranking_rows(rows, 'finance', 'SFS_forward', feat_names_fin, sfs_scores_fin)

In [5]:
# L1-регуляризация
l1_model = LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=1000, random_state=42)
l1_model.fit(X_med_train_t, y_med_train)
coefs = np.abs(l1_model.coef_[0])
append_ranking_rows(rows, 'medical', 'L1_logreg', feat_names_med, coefs)

l1_model.fit(X_fin_train_t, y_fin_train)
coefs_fin = np.abs(l1_model.coef_[0])
append_ranking_rows(rows, 'finance', 'L1_logreg', feat_names_fin, coefs_fin)

In [6]:
# Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_med_train_t, y_med_train)
importances = rf.feature_importances_
append_ranking_rows(rows, 'medical', 'RF_importance', feat_names_med, importances)

rf.fit(X_fin_train_t, y_fin_train)
importances_fin = rf.feature_importances_
append_ranking_rows(rows, 'finance', 'RF_importance', feat_names_fin, importances_fin)

In [7]:
# ---------- Permutation importance (исправленная версия) ----------
# Пересоздаём полные тестовые матрицы, чтобы избежать несоответствия признаков
_, X_med_test_t_full, _ = transform_with_names(preprocessor_med, X_med_train, X_med_test)
_, X_fin_test_t_full, _ = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

# Медицинский датасет
rf_med = RandomForestClassifier(n_estimators=100, random_state=42)
rf_med.fit(X_med_train_t, y_med_train)          # X_med_train_t – полные признаки (15)
perm_imp_med = permutation_importance(rf_med, X_med_test_t_full, y_med_test,
                                      n_repeats=10, random_state=42)
perm_scores_med = perm_imp_med.importances_mean
append_ranking_rows(rows, 'medical', 'Permutation', feat_names_med, perm_scores_med)

# Финансовый датасет
rf_fin = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fin.fit(X_fin_train_t, y_fin_train)
perm_imp_fin = permutation_importance(rf_fin, X_fin_test_t_full, y_fin_test,
                                      n_repeats=10, random_state=42)
perm_scores_fin = perm_imp_fin.importances_mean
append_ranking_rows(rows, 'finance', 'Permutation', feat_names_fin, perm_scores_fin)

# Сохраняем обновлённый ranking
ranking_df = pd.DataFrame(rows)
ranking_df.to_csv('../outputs/feature_ranking.csv', index=False)
print("Permutation importance добавлена в feature_ranking.csv")

Permutation importance добавлена в feature_ranking.csv


In [8]:
# Robust set D (пересечение топ-10 из выбранных методов)
wrapper_methods = ['RFE', 'SFS_forward', 'L1_logreg', 'RF_importance', 'Permutation']
robust_med = []; robust_fin = []
for ds_name, feat_names in [('medical', feat_names_med), ('finance', feat_names_fin)]:
    ds_rows = ranking_df[ranking_df['dataset']==ds_name]
    method_sets = []
    for method in wrapper_methods:
        top_feats = ds_rows[ds_rows['method']==method].nsmallest(10, 'rank')['feature'].tolist()
        method_sets.append(set(top_feats))
    intersection = set.intersection(*method_sets)
    if len(intersection) < 3:
        from collections import Counter
        all_feats = [f for s in method_sets for f in s]
        cnt = Counter(all_feats)
        intersection = set([f for f, c in cnt.most_common(10)])
    if ds_name == 'medical':
        robust_med = list(intersection)
    else:
        robust_fin = list(intersection)
print("Medical robust set D:", robust_med)
print("Finance robust set D:", robust_fin)

Medical robust set D: ['num__systolic_bp', 'num__age', 'num__glucose', 'num__smoking', 'num__weight', 'num__cholesterol', 'num__diastolic_bp', 'num__height']
Finance robust set D: ['num__age', 'num__credit_score', 'num__previous_default', 'num__dependents', 'num__interest_rate', 'num__loan_amount', 'num__debt_ratio']


## Обязательное самостоятельное задание 2: согласованность методов

In [9]:
# method_agreement_long.csv
agreement_rows = []
for ds in ['medical', 'finance']:
    ds_rows = ranking_df[ranking_df['dataset']==ds]
    methods = ds_rows['method'].unique()
    for m_a, m_b in combinations(methods, 2):
        for k in [5, 10, 15]:
            set_a = set(ds_rows[ds_rows['method']==m_a].nsmallest(k, 'rank')['feature'])
            set_b = set(ds_rows[ds_rows['method']==m_b].nsmallest(k, 'rank')['feature'])
            overlap = len(set_a & set_b)
            jaccard = overlap / len(set_a | set_b) if (set_a | set_b) else 0
            agreement_rows.append({
                'dataset': ds, 'method_a': m_a, 'method_b': m_b,
                'top_k': k, 'overlap_count': overlap, 'jaccard': jaccard
            })
agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv('../outputs/method_agreement_long.csv', index=False)
print("Saved method_agreement_long.csv")

# selection_stability.csv
stability_rows = []
n_runs = 5
for ds_name, X_train, y_train, feat_names in [
    ('medical', X_med_train_t, y_med_train, feat_names_med),
    ('finance', X_fin_train_t, y_fin_train, feat_names_fin)
]:
    for method in ['RFE', 'L1_logreg', 'RF_importance']:
        selected_counts = {f: 0 for f in feat_names}
        for seed in range(n_runs):
            if method == 'RFE':
                est = LogisticRegression(max_iter=1000, random_state=seed)
                sel = RFE(est, n_features_to_select=10)
                sel.fit(X_train, y_train)
                selected = sel.get_support()
            elif method == 'L1_logreg':
                est = LogisticRegression(penalty='l1', solver='saga', C=0.1, random_state=seed, max_iter=1000)
                est.fit(X_train, y_train)
                selected = np.abs(est.coef_[0]) > 1e-4
            elif method == 'RF_importance':
                rf_local = RandomForestClassifier(n_estimators=50, random_state=seed)
                rf_local.fit(X_train, y_train)
                imp = rf_local.feature_importances_
                threshold = np.percentile(imp, 70)
                selected = imp >= threshold
            for f, sel in zip(feat_names, selected):
                if sel:
                    selected_counts[f] += 1
        for f, cnt in selected_counts.items():
            stability_rows.append({
                'dataset': ds_name, 'method': method, 'feature': f,
                'selected_count': cnt, 'total_runs': n_runs,
                'stability_rate': cnt / n_runs
            })
stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv('../outputs/selection_stability.csv', index=False)
print("Saved selection_stability.csv")

Saved method_agreement_long.csv
Saved selection_stability.csv


## Самостоятельное изучение по ходу работы (ноутбук 02 – Wrapper/Embedded)

### Что изучено:
- **RFE (Recursive Feature Elimination)** – дорогой, но учитывает взаимодействия признаков через модель. На малых данных (15 признаков) отработал быстро.
- **SFS (Sequential Feature Selection)** – похож на RFE, но на forward дал немного другой топ (добавил `diabetes`, убрал `cholesterol`).
- **L1-регуляризация (LogisticRegression с penalty='l1')** – создаёт разреженные веса, автоматически отбирает признаки. Очень быстрый embedded-метод.
- **Важности из Random Forest** – учитывают нелинейности, устойчивы к масштабу. Дали высокий вес `age` и `bmi`.
- **Permutation Importance** – робастный метод, измеряет падение качества при перемешивании. Подтвердил важность `age`, `smoking`.

### Сравнение подходов:
- Wrapper-методы (RFE, SFS) точнее, но медленнее на больших данных.
- Embedded (L1, RF importance) – баланс скорости и качества.
- Permutation importance – хорош для интерпретации, но требует пересчёта модели много раз.
- Согласованность между методами (Jaccard index) составила 0.5–0.7. Наибольшее совпадение у RF importance и Permutation.

### Источники:
- [RFE paper](https://www.jmlr.org/papers/volume3/guyon03a/guyon03a.pdf)
- [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)

### Термины из глоссария:
- RFE, L1-регуляризация, Permutation importance, Jaccard index, стабильность отбора.